# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library. We follow the Croissant schema to reference entities by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via the Croissant schema URL, ensuring semantic structure and rich metadata.

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, their `@id`, names, and their fields and field `@id`s.

In [ ]:
# List record sets, fields, and field columns using their @id
rs_objs = getattr(metadata, 'record_set', [])

record_set_ids = []
all_record_sets = getattr(metadata, 'record_set', [])
if not all_record_sets:
    # If .record_set is empty, use top-level distributions as record sets
    all_record_sets = getattr(metadata, 'distribution', [])

print('Available record sets:')
for rset in all_record_sets:
    rset_id = getattr(rset, '@id', None) if hasattr(rset, '@id') else rset.get('@id', None)
    rset_name = getattr(rset, 'name', None) if hasattr(rset, 'name') else rset.get('name', None)
    # Some datasets have fields or columns nested
    fields = getattr(rset, 'field', None) if hasattr(rset, 'field') else rset.get('field', None)
    columns = getattr(rset, 'column', None) if hasattr(rset, 'column') else rset.get('column', None)
    print(f'- Record Set @id: {rset_id}, Name: {rset_name}')
    record_set_ids.append(rset_id)
    # Display fields/columns with their @id
    if fields:
        print('  Fields:')
        for fld in fields:
            fld_id = getattr(fld, '@id', None) if hasattr(fld, '@id') else fld.get('@id', None)
            fld_name = getattr(fld, 'name', None) if hasattr(fld, 'name') else fld.get('name', None)
            print(f'    - {fld_id}: {fld_name}')
    if columns:
        print('  Columns:')
        for col in columns:
            col_id = getattr(col, '@id', None) if hasattr(col, '@id') else col.get('@id', None)
            col_name = getattr(col, 'name', None) if hasattr(col, 'name') else col.get('name', None)
            print(f'    - {col_id}: {col_name}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We reference all Croissant entities by their `@id` for clarity.

We will extract the first record set as the main example (feel free to modify to other available record set `@id`s).

In [ ]:
# Extract records from each record set and load them as DataFrames
dataframes = {}

# Use the first record set for primary demonstration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    raise RuntimeError('No record sets found.')

for rset_id in record_set_ids:
    # mlcroissant expects the record_set parameter as the `@id` of the record set
    records = list(dataset.records(record_set=rset_id))
    if records:
        dataframes[rset_id] = pd.DataFrame(records)

print(f'Columns for main record set (@id: {main_record_set_id}):')
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping data. All fields are referenced by column name, which matches their Croissant `@id` when using the records extractor.

In [ ]:
# Determine possible numeric fields to analyze
df = dataframes[main_record_set_id]

numeric_field_ids = df.select_dtypes(include='number').columns.tolist()

print('Numeric fields available:', numeric_field_ids)
if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    # Example: Threshold is arbitrary, just for demo (choose 80th percentile)
    threshold = float(df[numeric_field].quantile(0.8))
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records (where {numeric_field} > {threshold}):")
    print(filtered_df.head())
    
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())
    
    # Now group by a non-numeric column if available
    group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (mean of numeric fields):")
        print(grouped_df.head())
else:
    print('No numeric fields available in the main record set.')

## 5. Visualization
Visualize the distribution of a numeric field and the relationship between two fields in the selected record set. All columns are referenced by their Croissant `@id`/column name.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_ids:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # If there is more than one numeric field, show their pairplot
    if len(numeric_field_ids) > 1:
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=df[numeric_field_ids[0]], y=df[numeric_field_ids[1]])
        plt.title(f"{numeric_field_ids[0]} vs {numeric_field_ids[1]}")
        plt.xlabel(numeric_field_ids[0])
        plt.ylabel(numeric_field_ids[1])
        plt.show()

## 6. Conclusion
We have used `mlcroissant` to explore the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset with FAIR² packaging. This included loading the dataset metadata, inspecting available record sets and fields by their `@id`, loading records into DataFrames, conducting basic filtering, normalization, and visualization.

For further analysis, consider examining field definitions in the Croissant schema and integrating downstream machine learning workflows based on these FAIR record set and field identifiers.